In [35]:
import os
import time
from matplotlib import pyplot as plt
import seaborn as sns
from scr.qsvdd_core.data_loader import QuantumDataLoader
import numpy as np
from ucimlrepo import fetch_ucirepo
import matplotlib.pyplot as plt
from scripts.train_model import circuit_training, train_five_times
from scr.qsvdd_core.data_loader import QuantumDataLoader
from scripts.test_model import test, mean_auc, best_batch, save_test_results

In [36]:
np.random.seed(42)

In [45]:
n_train = 0 ; latent_dim = 3
# num_params_conv = 375
cost_func = 'svdd'
learning_rate = 0.01

# Breast Cancer anomaly Detection

In [38]:
print("="*60)
print("Loading and processing the Breast Cancer dataset...")
print("="*60)

# 1. Fetch the dataset
breast_cancer = fetch_ucirepo(id=17)
X_bc_raw = breast_cancer.data.features
y_bc_raw = breast_cancer.data.targets

Loading and processing the Breast Cancer dataset...


In [ ]:
print(breast_cancer.data.features.head())

In [ ]:
X_bc_raw.info()

In [ ]:
y_bc_raw.info()

In [27]:
y_bc = y_bc_raw.iloc[:, 0].apply(lambda x: 1 if x == 'M' else 0).values

loader = QuantumDataLoader()

import pandas as pd
df_bc = pd.DataFrame(X_bc_raw)
df_bc['Class'] = y_bc

X_data = loader.prepare_bc_data(X_bc_raw)
y = y_bc

# 4. Identify indices for each class
normal_indices = np.where(y == 0)[0]
abnormal_indices = np.where(y == 1)[0]
np.random.seed(42)

# 5. Training Set (One-Class: 200 normal samples)
X_train_normal_indices = np.random.choice(normal_indices, 250, replace=False)
X_train = X_data[X_train_normal_indices]
Y_train = y[X_train_normal_indices]

# 6. Test Set (Balanced: 50 normal + 50 anomalies)
remaining_normal_indices = list(set(normal_indices) - set(X_train_normal_indices))

X_test_normal_indices = np.random.choice(remaining_normal_indices, 50, replace=False)
X_test_abnormal_indices = np.random.choice(abnormal_indices, 50, replace=False)

X_test_normal = X_data[X_test_normal_indices]
y_test_normal = y[X_test_normal_indices]

X_test_abnormal = X_data[X_test_abnormal_indices]
y_test_abnormal = y[X_test_abnormal_indices]

# Final test set (Mix: 100 samples)
X_test = np.concatenate((X_test_normal, X_test_abnormal), axis=0)
Y_test = np.concatenate((y_test_normal, y_test_abnormal), axis=0)

print(f"X_train shape (Normal): {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"X_test shape (Mix): {X_test.shape}")
print(f"Y_test shape: {Y_test.shape}")

X_train shape (Normal): (250, 32)
Y_train shape: (250,)
X_test shape (Mix): (100, 32)
Y_test shape: (100,)


In [28]:
loader = QuantumDataLoader()

X_quantum = loader.prepare_bc_data(X_bc_raw)

print(f"Features for the circuit: {X_quantum.shape}")
print(f"Labels: {y.shape}")

Features for the circuit: (569, 32)
Labels: (569,)


In [29]:
"""
One-class Training:
Separation into normal and fraudulent examples
QSVDD will learn what is normal.
"""

normal_indices = np.where(y == 0)[0]
abnormal_indices = np.where(y == 1)[0]
np.random.seed(42)

# Train
# collect 1000 normal examples for training
X_train_normal_indices = np.random.choice(normal_indices, 250, replace=False)
X_train_normal = X_quantum[X_train_normal_indices]
y_train_normal = y[X_train_normal_indices]

# X_train contains only legitimate transactions
X_train = X_train_normal
Y_train = y_train_normal

# Test (balanced)
# The code removes 100 normal examples that were not used in training
remaining_normal_indices = list(set(normal_indices) - set(X_train_normal_indices))
X_test_normal_indices = np.random.choice(remaining_normal_indices, 100, replace=False)
X_test_normal = X_quantum[X_test_normal_indices]
y_test_normal = y[X_test_normal_indices]

# The code removes 100 fraud examples.
X_test_abnormal_indices = np.random.choice(abnormal_indices, 100, replace=False)
X_test_abnormal = X_quantum[X_test_abnormal_indices]
y_test_abnormal = y[X_test_abnormal_indices]

# creates a test set with 200 examples (50% normal, 50% fraud)
X_test = np.concatenate((X_test_normal, X_test_abnormal), axis=0)
Y_test = np.concatenate((y_test_normal, y_test_abnormal), axis=0)

center = np.zeros(latent_dim)
center_train = np.tile(center, (len(X_train), 1))

print(f'X_train shape: {X_train.shape}')
print(f'Y_train shape: {Y_train.shape}')
print(f'X_test_normal shape: {X_test_normal.shape}')
print(f'X_test_abnormal shape: {X_test_abnormal.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Y_test shape: {Y_test.shape}')
print(f'center_train shape: {center_train.shape}')

X_train shape: (250, 32)
Y_train shape: (250,)
X_test_normal shape: (100, 32)
X_test_abnormal shape: (100, 32)
X_test shape: (200, 32)
Y_test shape: (200,)
center_train shape: (250, 3)


In [30]:
train_Xdata = X_train
train_Ydata = center_train

### QCNN (Quantum Convolutional Neural Network) Ansatz

#### Hyperparameters

In [31]:
qcnn_batch_size = 4
qcnn_steps = 2000

In [33]:
(qcnn_loss_history_matrix,
 qcnn_est_params_matrix,
 qcnn_param_history_matrix,
 qcnn_time_record) = train_five_times(X_train=train_Xdata,
                                        Y_train=train_Ydata,
                                        batch_size=qcnn_batch_size,
                                        learning_rate=learning_rate,
                                        steps=qcnn_steps,
                                        ansatz='qcnn'
                                        )
loss_history_f_name = f"../results/training/QCNN/BC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{learning_rate:.0e}_LOSS_HISTORY_MEAN.npy"
est_params_f_name = f"../results/training/QCNN/BC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{learning_rate:.0e}_EST_PARAMS_MEAN.npy"
time_f_name = f"../results/training/QCNN/BC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{learning_rate:.0e}_TIME_MEAN.npy"
np.savetxt(loss_history_f_name, qcnn_loss_history_matrix)
np.savetxt(est_params_f_name, qcnn_est_params_matrix)
np.savetxt(time_f_name, qcnn_time_record)
print("--- All training batches completed ---")

--- Starting training round 1 with seed 1848857951 ---


/home/jvfg/Documents/ORG/Repos/QSVDD2/.venv/lib/python3.12/site-packages/autograd/numpy/numpy_vjps.py:943: ComplexWarning: Casting complex values to real discards the imaginary part
  onp.add.at(A, idx, x)


--- Starting training round 2 with seed 452643366 ---
--- Starting training round 3 with seed 3619198621 ---
--- Starting training round 4 with seed 1328972624 ---
--- Starting training round 5 with seed 919471137 ---
--- All training batches completed ---


#### QCNN Training Evaluation

In [34]:
f_name = f"../results/training/QCNN/BC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{learning_rate:.0e}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="qcnn")

print(50*"--")
print(f'for: B{qcnn_batch_size}S{qcnn_steps} | AUC_mean: {mean} | std: {std}')
print(50*"--")

Processing class 0 (label 0) | Samples: 100
Finished class 0 in 2.59s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 2.30s
Test completed in 4.90s | AUC: 0.7269
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 2.31s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 2.53s
Test completed in 4.84s | AUC: 0.3833
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 2.22s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 2.20s
Test completed in 4.42s | AUC: 0.5598
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 2.51s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 2.29s
Test completed in 4.81s | AUC: 0.5979
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 2.25s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 2.48s
Test completed in 4.73s | AUC: 0.7384
----------------------------------------------------------------------------------------------------
for: B4S2

### QAE (Quantum AutoEncoder) Ansatz

In [40]:
qae_batch_size = 16
qae_steps = 500

#### Five-Run Training & Result Persistence

Run `train_five_times` to train the **QAE** ansatz across 5 independent runs with the configured batch size and step count.

In [41]:
(qae_loss_history_matrix,
 qae_est_params_matrix,
 qae_param_history_matrix,
 qae_time_record) = train_five_times(X_train=train_Xdata,
                                        Y_train=train_Ydata,
                                        batch_size=qae_batch_size,
                                        learning_rate=learning_rate,
                                        steps=qae_steps,
                                        ansatz='qae'
                                        )
loss_history_f_name = f"../results/training/QAE/BC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{learning_rate:.0e}_LOSS_HISTORY_MEAN.npy"
est_params_f_name = f"../results/training/QAE/BC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{learning_rate:.0e}_EST_PARAMS_MEAN.npy"
time_f_name = f"../results/training/QAE/BC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{learning_rate:.0e}_TIME_MEAN.npy"
np.savetxt(loss_history_f_name, qae_loss_history_matrix)
np.savetxt(est_params_f_name, qae_est_params_matrix)
np.savetxt(time_f_name, qae_time_record)
print("--- All training batches completed ---")

--- Starting training round 1 with seed 2707504392 ---
--- Starting training round 2 with seed 246080965 ---
--- Starting training round 3 with seed 3991555978 ---
--- Starting training round 4 with seed 2367246785 ---
--- Starting training round 5 with seed 689508831 ---
--- All training batches completed ---


#### QAE Training Evaluation

In [43]:
f_name = f"../results/training/QAE/BC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{learning_rate:.0e}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="qae")

print(50*"--")
print(f'for: B{qae_batch_size}S{qae_steps} | AUC_mean: {mean} | std: {std}')
print(50*"--")

Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.18s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.14s
Test completed in 2.33s | AUC: 0.6583
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.14s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.14s
Test completed in 2.28s | AUC: 0.7904
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.14s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.15s
Test completed in 2.29s | AUC: 0.8496
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.16s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.15s
Test completed in 2.31s | AUC: 0.3881
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 1.16s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 1.16s
Test completed in 2.33s | AUC: 0.7812
----------------------------------------------------------------------------------------------------
for: B16S

### LCQHNN (Lean classical-quantum hybrid neural network) Ansatz

The latent space for this ansatz has dimension 5 (vs. 3 for QCNN/QAE), requiring a re-initialized center vector.

In [46]:
center = np.zeros(5)
center_train = np.tile(center, (len(X_train), 1))
lcqhnn_batch_size = 8
lcqhnn_steps = 1000
print(f'center_train shape: {center_train.shape}')
train_Xdata = X_train
train_Ydata = center_train

center_train shape: (250, 5)


#### Five-Run Training & Result Persistence

Run `train_five_times` to train the **QAE** ansatz across 5 independent runs with the configured batch size and step count.

In [47]:
(lcqhnn_loss_history_matrix,
 lcqhnn_est_params_matrix,
 lcqhnn_param_history_matrix,
 lcqhnn_time_record) = train_five_times(X_train=train_Xdata,
                                        Y_train=train_Ydata,
                                        batch_size=lcqhnn_batch_size,
                                        learning_rate=learning_rate,
                                        steps=lcqhnn_steps,
                                        ansatz='lcqhnn'
                                        )
loss_history_f_name = f"../results/training/LCQHNN/BC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{learning_rate:.0e}_LOSS_HISTORY_MEAN.npy"
est_params_f_name = f"../results/training/LCQHNN/BC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{learning_rate:.0e}_EST_PARAMS_MEAN.npy"
time_f_name = f"../results/training/LCQHNN/BC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{learning_rate:.0e}_TIME_MEAN.npy"
np.savetxt(loss_history_f_name, lcqhnn_loss_history_matrix)
np.savetxt(est_params_f_name, lcqhnn_est_params_matrix)
np.savetxt(time_f_name, lcqhnn_time_record)
print("--- All training batches completed ---")

--- Starting training round 1 with seed 2510952096 ---
--- Starting training round 2 with seed 3620618533 ---
--- Starting training round 3 with seed 3894174595 ---
--- Starting training round 4 with seed 3000719540 ---
--- Starting training round 5 with seed 146229895 ---
--- All training batches completed ---


#### LCQHNN Training Evaluation

In [49]:
f_name = f"../results/training/LCQHNN/BC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{learning_rate:.0e}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="lcqhnn")

print(50*"--")
print(f'for: B{lcqhnn_batch_size}S{lcqhnn_steps} | AUC_mean: {mean} | std: {std}')
print(50*"--")

Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.21s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.24s
Test completed in 0.45s | AUC: 0.5914
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.22s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.25s
Test completed in 0.47s | AUC: 0.4339
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.22s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.23s
Test completed in 0.45s | AUC: 0.5331
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.27s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.23s
Test completed in 0.51s | AUC: 0.6536
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.22s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.24s
Test completed in 0.47s | AUC: 0.4465
----------------------------------------------------------------------------------------------------
for: B8S1

## Noisy (NISQ-Era) Training

In the noisy setting, the quantum circuits are simulated with a **hardware noise model** that mimics real NISQ (Noisy Intermediate-Scale Quantum) device behavior, including gate errors and decoherence. This evaluates model robustness under realistic quantum hardware conditions.

The same three ansatzes are retrained from scratch under this noisy simulation.